In [1]:
"""
accessibility calculation.
"""

'\naccessibility calculation.\n'

In [2]:
# ============================================================
# 0. Imports & Font Configuration
# ------------------------------------------------------------
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
     "pandas==2.2.2", "geopandas==1.0.1", "matplotlib==3.10.0","seaborn==0.13.2"])

import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import matplotlib.font_manager as fm
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.image as img
import time as time
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
os.chdir(os.getcwd())

Tnr_path = os.path.join(os.getcwd(), 'Times New Roman.ttf')
font_manager.fontManager.addfont(Tnr_path)
regular_prop = font_manager.FontProperties(fname=Tnr_path)

italic_path = os.path.join(os.getcwd(), 'Roman Italic.ttf')
font_manager.fontManager.addfont(italic_path)
italic_prop = font_manager.FontProperties(fname=italic_path)

plt.rcParams['font.family']      = 'serif'
plt.rcParams['font.serif']       = [regular_prop.get_name()]
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm']      = regular_prop.get_name()
plt.rcParams['mathtext.it']      = italic_prop.get_name()

In [3]:
# ============================================================
# 1. Load Aggregated Link & Node Layers
# ------------------------------------------------------------

cities_links = [f for f in os.listdir('data/groupedlinks/') if f.endswith('_link_betweenness.gpkg')]
part_cities_links = [gpd.read_file(os.path.join('data/groupedlinks/', file)) for file in cities_links]
all_cities_links = gpd.GeoDataFrame(pd.concat(part_cities_links, ignore_index=True))

cities_nodes = [f for f in os.listdir('data/groupednodes/') if f.endswith('_node_centrality.gpkg')]
part_cities_nodes = [gpd.read_file(os.path.join('data/groupednodes/', file)) for file in cities_nodes]
all_cities_nodes = gpd.GeoDataFrame(pd.concat(part_cities_nodes, ignore_index=True))
unique_sgg_nm = all_cities_links['SIGUNGU_NM'].dropna().unique()


In [4]:
# ============================================================
# 2. Helper Functions: North Arrow & Scale Bar
# ------------------------------------------------------------
def add_north_arrow(ax, x, y, arrow_file, zoom=0.05):
    im = img.imread(arrow_file)
    imagebox = OffsetImage(im, zoom=zoom)
    ab = AnnotationBbox(imagebox, (x, y), xycoords='axes fraction', frameon=False)
    ax.add_artist(ab)
    
def add_scale_bar(ax, length, location=(0.1, 0.02), linewidth=3, color='black'):
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    sb_x = xlim[0] + (xlim[1] - xlim[0]) * location[0]
    sb_y = ylim[0] + (ylim[1] - ylim[0]) * location[1]
    scale_bar_x = [sb_x, sb_x + length]
    scale_bar_y = [sb_y, sb_y]

    ax.plot(scale_bar_x, scale_bar_y, color=color, linewidth=linewidth)
    ax.text(sb_x + length / 2, sb_y, f'{round(length/1000):,} km', verticalalignment='bottom', horizontalalignment='center', fontsize=15, fontproperties=fm.FontProperties(size=10))

In [5]:
# ============================================================
# 3. Transportation Cost Calculation
# ------------------------------------------------------------

if not os.path.exists('data/plot/map_cost/'):
    os.makedirs('data/plot/map_cost/')
    
def scaler(df, column):
    return df[column].rank(pct=True)
def formatter(x, pos):
    return f'{x:.01f}'

midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')

processing_city = 0

for city in unique_sgg_nm:
    #if os.path.exists(f'data/plot/map_straightness/straightness_{city}.png'):
    #    print(f'{city} already exists, skipping. {processing_city}/{len(unique_sgg_nm)}')
    #    continue
    
    city_boundary = midsize_cities[midsize_cities['SIGUNGU_NM'] == city].copy()
    city_link = all_cities_links[all_cities_links['SIGUNGU_NM'] == city].copy()
    city_node = all_cities_nodes[all_cities_nodes['SIGUNGU_NM'] == city].copy()
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    city_NM_EN = midsize_cities[midsize_cities['SIGUNGU_NM'] == city]['SIGUNGU_NM_EN'].values[0]
    fig.suptitle(f'{city_NM_EN}\nTransportation cost', fontsize=20, fontproperties=regular_prop, y=0.95)

    # Standardize the betweenness centrality values
    city_node['cost'] = city_node['closeness'].copy()
    city_node['cost_scaled'] = scaler(city_node, 'cost')
    
    # Plot betweenness centrality
    city_link.plot(ax=ax, color='black',legend= False, linewidth = 0.7)
    legend_elements = [Line2D([0], [0], color='black', lw=2, label = 'Roads')]
    # Plot city boundary
    city_boundary.plot(ax=ax, color='none', edgecolor='black', linewidth=2, zorder=2)
    city_node.plot(column='cost_scaled', cmap='Reds', linewidth=0.5, ax=ax, legend=False, alpha=1)
    sm = plt.cm.ScalarMappable(cmap='Reds', norm=plt.Normalize(vmin=0.5, vmax=1.0))
    sm._A = []
    cbar = plt.colorbar(sm, ax=ax)
    cbar.ax.yaxis.set_major_formatter(FuncFormatter(formatter))
    
    # Set the x and y limits to fit the city boundary
    ax.legend(handles=legend_elements, loc='upper left', fontsize=15)
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_axis_off()
    # Add north arrow and scale bar
    add_north_arrow(ax, x=0.95, y=0.95, arrow_file='north_arrow.png', zoom=0.2)
    add_scale_bar(ax,location = (0.9, 0), length=5000)    
    plt.tight_layout()
    plt.savefig(f'data/plot/map_cost/cost_{city}.png')
    plt.close()

    mean_cost = city_node['cost'].mean()
    midsize_cities.loc[midsize_cities['SIGUNGU_NM'] == city, 'MTC'] = mean_cost/midsize_cities.loc[midsize_cities['SIGUNGU_NM'] == city, 'nodes']**0.5
    midsize_cities.loc[midsize_cities['SIGUNGU_NM'] == city, 'CVTC'] = city_node['cost'].std(ddof=0) / city_node['cost'].mean()
    

midsize_cities.to_file('data/midsize_cities_index.gpkg', driver='GPKG')

In [6]:
# ============================================================
# 4. detour rate calculation
# ------------------------------------------------------------

if not os.path.exists('data/plot/map_detour_rate/'):
    os.makedirs('data/plot/map_detour_rate/')
    
def scaler(df, column):
    return df[column].rank(pct=True)
def formatter(x, pos):
    return f'{x:.01f}'

midsize_cities = gpd.read_file('data/midsize_cities_index.gpkg')

processing_city = 0

for city in unique_sgg_nm:
    processing_city += 1
    start_time = time.time()
    #if os.path.exists(f'data/plot/map_straightness/straightness_{city}.png'):
    #    print(f'{city} already exists, skipping. {processing_city}/{len(unique_sgg_nm)}')
    #    continue
    
    city_boundary = midsize_cities[midsize_cities['SIGUNGU_NM'] == city].copy()
    city_link = all_cities_links[all_cities_links['SIGUNGU_NM'] == city].copy()
    city_node = all_cities_nodes[all_cities_nodes['SIGUNGU_NM'] == city].copy()
    
    fig, ax = plt.subplots(figsize=(8, 8))
    
    city_NM_EN = midsize_cities[midsize_cities['SIGUNGU_NM'] == city]['SIGUNGU_NM_EN'].values[0]
    fig.suptitle(f'{city_NM_EN}\n detour rate', fontsize=20, fontproperties=regular_prop, y=0.95)

    # Standardize the betweenness centrality values
    city_node['detour_rate'] = city_node['straightness_normalized'].copy()
    city_node['detour_rate_scaled'] = scaler(city_node, 'detour_rate')
    
    # Plot betweenness centrality
    city_link.plot(ax=ax, color='black',legend= False, linewidth = 0.7)
    legend_elements = [Line2D([0], [0], color='black', lw=2, label = 'Roads')]
    # Plot city boundary
    city_boundary.plot(ax=ax, color='none', edgecolor='black', linewidth=2, zorder=2)
    city_node.plot(column='detour_rate_scaled', cmap='Reds', linewidth=0.5, ax=ax, legend=False, alpha=1)
    sm = plt.cm.ScalarMappable(cmap='Reds', norm=plt.Normalize(vmin=0.5, vmax=1.0))
    sm._A = []
    cbar = plt.colorbar(sm, ax=ax)
    cbar.ax.yaxis.set_major_formatter(FuncFormatter(formatter))
    
    # Set the x and y limits to fit the city boundary
    ax.legend(handles=legend_elements, loc='upper left', fontsize=15)
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_axis_off()
    # Add north arrow and scale bar
    add_north_arrow(ax, x=0.95, y=0.95, arrow_file='north_arrow.png', zoom=0.2)
    add_scale_bar(ax,location = (0.9, 0), length=5000)    
    plt.tight_layout()
    plt.savefig(f'data/plot/map_detour_rate/detour_rate_{city}.png')
    plt.close()

    detour_rate = city_node['detour_rate'].mean()
    midsize_cities.loc[midsize_cities['SIGUNGU_NM'] == city, 'MDR'] = detour_rate
    midsize_cities.loc[midsize_cities['SIGUNGU_NM'] == city, 'CVDR'] = (city_node['detour_rate'].std(ddof=0)*(len(city_node)-1)**0.5 / city_node['detour_rate'].mean())

midsize_cities.to_file('data/midsize_cities_index.gpkg', driver='GPKG')